In [30]:
import os
from FID.fid_score import calculate_fid, get_precomputed_fid_scores_path, save_fid_stats_as_dict
import gzip
import numpy as np
import os
from PIL import Image
import pickle
import torch
import yaml
import argparse
from pathlib import Path
from utils.data_utils import get_data, get_gen
from utils.utils import reset_random_seeds, prepare_config
from models.losses import loss_reconstruction_binary, loss_reconstruction_mse

In [21]:
path = '/Users/jorgegoncalves/Library/CloudStorage/OneDrive-Persönlich/Dokumente/Universität/Master/HS23/Master_Thesis/Code/treevae/models/experiments/'

dataset = 'mnist/'
ex_name = '20240221-150914_9e40a'

checkpoint_path = path+dataset+ex_name
with open(checkpoint_path + "/config.yaml", 'r') as stream:
    configs = yaml.load(stream,Loader=yaml.Loader)
print(configs)

from utils.data_utils import get_data, get_gen

# load data
trainset, trainset_eval, testset = get_data(configs)
gen_train = get_gen(trainset, configs, validation=False, shuffle=False)
gen_train_eval = get_gen(trainset_eval, configs, validation=True, shuffle=False)
gen_test = get_gen(testset, configs, validation=True, shuffle=False)
gen_train_eval_iter = iter(gen_train_eval)
gen_test_iter = iter(gen_test)
y_train = trainset_eval.dataset.targets.clone().detach()[trainset_eval.indices].numpy()
y_test = testset.dataset.targets.clone().detach()[testset.indices].numpy()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device = "mps"

{'data': {'data_name': 'mnist', 'num_clusters_data': 10}, 'ddpm': {'data': {'data_name': 'mnist', 'ddpm_latent_path': '', 'image_size': 28, 'inp_channels': 1, 'norm': True, 'num_clusters_data': 10}, 'evaluation': {'chkpt_path': 'results/mnist/checkpoints/ddpmv2-vae-epoch=1026-loss=0.0064.ckpt', 'eval_mode': 'recons_all_leaves', 'guidance_weight': 0.0, 'n_samples': 10000, 'n_steps': 1000, 'resample_strategy': 'truncated', 'sample_from': 'target', 'sample_method': 'ddpm', 'sample_prefix': '', 'save_mode': 'image', 'save_path': 'results/mnist/', 'save_vae': True, 'seed': 0, 'skip_strategy': 'quad', 'temp': 1.0, 'type': 'form1', 'variance': 'fixedlarge', 'z_cond': True, 'z_dim': 1}, 'globals': {'seed': 42}, 'model': {'attn_resolutions': '16,', 'beta1': 0.0001, 'beta2': 0.02, 'dim': 64, 'dim_mults': '1,2,2,2', 'dropout': 0.3, 'n_heads': 8, 'n_residual': 2, 'n_timesteps': 1000}, 'training': {'batch_size': 256, 'cfd_rate': 0.0, 'chkpt_interval': 1, 'chkpt_prefix': 'vae', 'ema_decay': 0.9999, 

In [8]:
ddpm_samples_path = 'results/' + dataset + 'ddpm/sample/'
ddpm_reconstructions_path = 'results/' + dataset + 'ddpm/recons/'

vae_samples_path = 'results/' + dataset + 'vae/sample/'
vae_reconstructions_path = 'results/' + dataset + 'vae/recons/'

In [19]:
# load all images from the path and create a numpy array
ddpm_samples = []
for filename in os.listdir(ddpm_samples_path):
    if filename.endswith(".png"):
        img = Image.open(ddpm_samples_path + filename)
        img = np.array(img)
        ddpm_samples.append(img)
ddpm_samples = np.array(ddpm_samples)
print("Nb. of ddpm samples: ", len(ddpm_samples), "\n")

ddpm_reconstructions = []
for filename in os.listdir(ddpm_reconstructions_path):
    if filename.endswith(".png"):
        img = Image.open(ddpm_reconstructions_path + filename)
        img = np.array(img)
        ddpm_reconstructions.append(img)
ddpm_reconstructions = np.array(ddpm_reconstructions)
print("Nb. of ddpm reconstructions: ", len(ddpm_reconstructions), "\n")

vae_samples = []
for filename in os.listdir(vae_samples_path):
    if filename.endswith(".png"):
        img = Image.open(vae_samples_path + filename)
        img = np.array(img)
        vae_samples.append(img)
vae_samples = np.array(vae_samples)
print("Nb. of vae samples: ", len(vae_samples), "\n")

vae_reconstructions = []
for filename in os.listdir(vae_reconstructions_path):
    if filename.endswith(".png"):
        img = Image.open(vae_reconstructions_path + filename)
        img = np.array(img)
        vae_reconstructions.append(img)
vae_reconstructions = np.array(vae_reconstructions)
print("Nb. of vae reconstructions: ", len(vae_reconstructions), "\n")



Nb. of ddpm samples:  256 

Nb. of ddpm reconstructions:  256 

Nb. of vae samples:  256 

Nb. of vae reconstructions:  256 



In [22]:
   # precompute or load fid scores for train and test
data_stats_train = get_precomputed_fid_scores_path(trainset.dataset.data, configs['data']['data_name'],
                                                subset="train", device=device)
data_stats_test = get_precomputed_fid_scores_path(testset.dataset.data, configs['data']['data_name'], subset="test",
                                                device=device)

---

VAE SAMPLES

In [25]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(vae_samples, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)

Saving FID statistics


100%|██████████| 1/1 [00:10<00:00, 10.87s/it]


FID score for generated images compared to train set: 258.4713836317428
FID score for generated images compared to test set: 258.2609257564007


DDPM SAMPLES

In [26]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(ddpm_samples, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)


Saving FID statistics


100%|██████████| 1/1 [00:11<00:00, 11.24s/it]


FID score for generated images compared to train set: 404.2060109709134
FID score for generated images compared to test set: 405.51761674508737


---

VAE RECONSTRUCTIONS

In [27]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(vae_reconstructions, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)

Saving FID statistics


100%|██████████| 1/1 [00:10<00:00, 10.63s/it]


FID score for generated images compared to train set: 210.82774306285893
FID score for generated images compared to test set: 210.57307943768757


DDPM RECONSTRUCTIONS

In [28]:
# precompute FID scores for generated images
stats_generations = save_fid_stats_as_dict(ddpm_reconstructions, batch_size=256, device=device, dims=2048)
train_FID_generations = calculate_fid([data_stats_train, stats_generations], batch_size=256, device=device, dims=2048)
test_FID_generations = calculate_fid([data_stats_test, stats_generations], batch_size=256, device=device, dims=2048)
print("FID score for generated images compared to train set:", train_FID_generations)
print("FID score for generated images compared to test set:", test_FID_generations)

Saving FID statistics


100%|██████████| 1/1 [00:08<00:00,  8.79s/it]


FID score for generated images compared to train set: 404.29560675336353
FID score for generated images compared to test set: 405.60730663250285


---